In [ ]:
# zelle 1

In [ ]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml


def find_project_root(start: Path) -> Path:
    """
    Suche vom aktuellen Ordner aus nach oben nach einem
    Projektordner, der src/walinet enthält.
    """
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (candidate / "src" / "walinet").is_dir():
            return candidate

    raise FileNotFoundError(
        "WALINET project root not found. "
        "Start the notebook somewhere inside the repository "
        "or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = find_project_root(
    Path.cwd()
)

src_dir = PROJECT_ROOT / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(
        0,
        str(src_dir),
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src directory:", src_dir)

In [ ]:
TRAIN_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "Training"
    / "train_7T.yaml"
)


print("Training config:")
print(TRAIN_CONFIG_PATH)
print("Exists:", TRAIN_CONFIG_PATH.is_file())


assert TRAIN_CONFIG_PATH.is_file(), (
    "Training config not found:\n"
    f"{TRAIN_CONFIG_PATH}"
)

In [ ]:
from walinet.training_data.build_simulation_system import (
    build_simulation_system,
)


system = build_simulation_system(
    TRAIN_CONFIG_PATH
)

train_cfg = system.train_config
simulation_cfg = system.simulation_config


print(
    "Complete simulation system loaded "
    "and validated successfully."
)

print()
print("Training config:")
print(system.train_config_path)

print()
print("Simulation config:")
print(system.simulation_config_path)

print()
print("Basis library:")
print(simulation_cfg.basis.library)

print()
print("Metabolite profiles:")

for index, profile in enumerate(
    simulation_cfg.metabolites.profiles
):
    print(
        f"  [{index}] "
        f"{profile.config}"
    )
    print(
        f"      probability = "
        f"{profile.probability:.3f}"
    )

print()
print("Frequency shift:")
print(
    f"  mean = "
    f"{simulation_cfg.metabolites.frequency_shift.mean_hz:.3f} Hz"
)
print(
    f"  std  = "
    f"{simulation_cfg.metabolites.frequency_shift.std_hz:.3f} Hz"
)

print()
print("Voigt FWHM:")
print(
    f"  mean = "
    f"{simulation_cfg.metabolites.fwhm.mean_hz:.3f} Hz"
)
print(
    f"  std  = "
    f"{simulation_cfg.metabolites.fwhm.std_hz:.3f} Hz"
)

print()
print("Water scaling:")
print(
    f"  mean = "
    f"{simulation_cfg.water.scaling_mean:.3f}"
)
print(
    f"  std  = "
    f"{simulation_cfg.water.scaling_std:.3f}"
)

print()
print("Lipid scaling:")
print(
    f"  min = "
    f"{simulation_cfg.lipids.scaling_min:.3f}"
)
print(
    f"  max = "
    f"{simulation_cfg.lipids.scaling_max:.3f}"
)

print()
print("Device:")
print(system.device)

In [ ]:
print("TRAINING CONFIG")
print("=" * 60)

print(
    "Run name:",
    train_cfg.run.name,
)

print(
    "Data source:",
    train_cfg.data.source,
)

print(
    "Base directory:",
    train_cfg.data.base_dir,
)

print(
    "Train subjects:",
    len(
        train_cfg.data.train_subjects
    ),
)

print(
    "Validation subjects:",
    len(
        train_cfg.data.val_subjects
    ),
)

print(
    "Normalization:",
    train_cfg.data.normalization,
)

print(
    "Training batch size:",
    train_cfg.training.batch_size,
)

print(
    "Validation spectra:",
    train_cfg.validation.n_spectra,
)

print(
    "Validation seed:",
    train_cfg.validation.seed,
)


print()
print("SIMULATION CONFIG")
print("=" * 60)

print(
    "Version:",
    simulation_cfg.version,
)

print(
    "Bandwidth [Hz]:",
    simulation_cfg
    .acquisition
    .bandwidth_hz,
)

print(
    "Target timepoints:",
    simulation_cfg
    .acquisition
    .n_timepoints,
)

print(
    "NMR frequency [Hz]:",
    simulation_cfg
    .acquisition
    .nmr_frequency_hz,
)

print(
    "Subject mixing:",
    simulation_cfg
    .subject_sampling
    .mixing,
)

print(
    "Lipid projection enabled:",
    simulation_cfg
    .lipid_projection
    .enabled,
)

print(
    "Basis library:",
    simulation_cfg
    .basis
    .library,
)

print(
    "Metabolite profiles:",
)

for index, profile in enumerate(
    simulation_cfg
    .metabolites
    .profiles
):
    print(
        f"  [{index}] "
        f"{profile.config}"
    )
    print(
        f"      probability = "
        f"{profile.probability:.3f}"
    )

In [ ]:
assert (
    train_cfg.data.source
    == "on_the_fly"
)

assert (
    train_cfg.data.on_the_fly
    is not None
)

resource_template = (
    train_cfg
    .data
    .on_the_fly
    .resources
    .filename
)

resource_version = (
    train_cfg
    .data
    .on_the_fly
    .resources
    .version
)

resource_relative_path = Path(
    resource_template.format(
        version=resource_version,
    )
)

base_dir = Path(
    train_cfg.data.base_dir
)

print(
    "Relative resource path:",
    resource_relative_path,
)

print(
    "Base directory:",
    base_dir,
)

In [ ]:
def collect_resource_paths(
    subjects: list[str],
) -> pd.DataFrame:
    rows = []

    for subject in subjects:
        path = (
            base_dir
            / subject
            / resource_relative_path
        )

        rows.append(
            {
                "subject": subject,
                "resource_path": str(path),
                "exists": path.is_file(),
                "size_mb": (
                    path.stat().st_size
                    / 1024**2
                    if path.is_file()
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


train_paths = collect_resource_paths(
    train_cfg.data.train_subjects
)

validation_paths = (
    collect_resource_paths(
        train_cfg.data.val_subjects
    )
)

print("Training resources")
display(train_paths)

print("Validation resources")
display(validation_paths)


missing_train = train_paths.loc[
    ~train_paths["exists"],
    "subject",
].tolist()

missing_validation = (
    validation_paths.loc[
        ~validation_paths["exists"],
        "subject",
    ].tolist()
)

assert not missing_train, (
    "Missing training resources:\n"
    f"{missing_train}"
)

assert not missing_validation, (
    "Missing validation resources:\n"
    f"{missing_validation}"
)

print(
    "All expected resource files exist."
)

In [ ]:
from walinet.training_data.simulation_resources import (
    SimulationPool,
    SimulationResources,
    build_simulation_resources,
)


resources = build_simulation_resources(
    train_cfg=train_cfg,
    simulation_cfg=simulation_cfg,
)

print(
    "Resource pools created successfully."
)

In [ ]:
def tensor_size_mb(
    tensor: torch.Tensor | None,
) -> float:
    if tensor is None:
        return 0.0

    return (
        tensor.numel()
        * tensor.element_size()
        / 1024**2
    )

In [ ]:
def summarize_pool(
    name: str,
    pool: SimulationPool,
) -> None:
    print(name)
    print("=" * 60)

    print(
        "Subjects:",
        pool.n_subjects,
    )

    print(
        "Subject names:",
        pool.subject_names,
    )

    print(
        "Water spectra:",
        tuple(
            pool.water_spectra.shape
        ),
        pool.water_spectra.dtype,
    )

    print(
        "Lipid spectra:",
        tuple(
            pool.lipid_spectra.shape
        ),
        pool.lipid_spectra.dtype,
    )

    print(
        "Water offsets:",
        tuple(
            pool.water_offsets.shape
        ),
    )

    print(
        "Lipid offsets:",
        tuple(
            pool.lipid_offsets.shape
        ),
    )

    print(
        "Water counts per subject:",
        pool.water_counts.tolist(),
    )

    print(
        "Lipid counts per subject:",
        pool.lipid_counts.tolist(),
    )

    print(
        "Native lengths:",
        pool.native_lengths.tolist(),
    )

    print(
        "Bandwidth [Hz]:",
        pool.bandwidth_hz,
    )

    print(
        "Target spectral points:",
        pool.n_timepoints,
    )

    print(
        "Device:",
        pool.device,
    )

    print(
        "Water memory [MB]:",
        round(
            tensor_size_mb(
                pool.water_spectra
            ),
            2,
        ),
    )

    print(
        "Lipid memory [MB]:",
        round(
            tensor_size_mb(
                pool.lipid_spectra
            ),
            2,
        ),
    )

    print(
        "Projection memory [MB]:",
        round(
            tensor_size_mb(
                pool.lipid_projection_operators
            ),
            2,
        ),
    )

    total_memory = (
        tensor_size_mb(
            pool.water_spectra
        )
        + tensor_size_mb(
            pool.lipid_spectra
        )
        + tensor_size_mb(
            pool.lipid_projection_operators
        )
    )

    print(
        "Total main tensor memory [MB]:",
        round(
            total_memory,
            2,
        ),
    )

    print()


summarize_pool(
    "TRAIN POOL",
    resources.train,
)

summarize_pool(
    "VALIDATION POOL",
    resources.validation,
)

In [ ]:
def validate_pool(
    pool: SimulationPool,
    expected_subjects: list[str],
) -> None:
    target_t = (
        simulation_cfg
        .acquisition
        .n_timepoints
    )

    expected_bandwidth = (
        simulation_cfg
        .acquisition
        .bandwidth_hz
    )

    assert (
        pool.subject_names
        == tuple(expected_subjects)
    )

    assert (
        pool.n_subjects
        == len(expected_subjects)
    )

    assert pool.water_spectra.ndim == 2
    assert pool.lipid_spectra.ndim == 2

    assert (
        pool.water_spectra.shape[1]
        == target_t
    )

    assert (
        pool.lipid_spectra.shape[1]
        == target_t
    )

    assert (
        pool.water_spectra.dtype
        == torch.complex64
    )

    assert (
        pool.lipid_spectra.dtype
        == torch.complex64
    )

    assert (
        pool.water_offsets.dtype
        == torch.int64
    )

    assert (
        pool.lipid_offsets.dtype
        == torch.int64
    )

    assert (
        pool.native_lengths.dtype
        == torch.int64
    )

    assert (
        pool.water_offsets.shape
        == (pool.n_subjects + 1,)
    )

    assert (
        pool.lipid_offsets.shape
        == (pool.n_subjects + 1,)
    )

    assert (
        pool.native_lengths.shape
        == (pool.n_subjects,)
    )

    assert (
        int(
            pool.water_offsets[0]
        )
        == 0
    )

    assert (
        int(
            pool.lipid_offsets[0]
        )
        == 0
    )

    assert torch.all(
        pool.water_offsets[1:]
        >= pool.water_offsets[:-1]
    )

    assert torch.all(
        pool.lipid_offsets[1:]
        >= pool.lipid_offsets[:-1]
    )

    assert (
        int(
            pool.water_offsets[-1]
        )
        == pool.n_water_spectra
    )

    assert (
        int(
            pool.lipid_offsets[-1]
        )
        == pool.n_lipid_spectra
    )

    assert torch.all(
        pool.water_counts > 0
    )

    assert torch.all(
        pool.lipid_counts > 0
    )

    assert torch.all(
        pool.native_lengths > 0
    )

    assert torch.isfinite(
        pool.water_spectra.real
    ).all()

    assert torch.isfinite(
        pool.water_spectra.imag
    ).all()

    assert torch.isfinite(
        pool.lipid_spectra.real
    ).all()

    assert torch.isfinite(
        pool.lipid_spectra.imag
    ).all()

    assert not torch.any(
        torch.all(
            pool.water_spectra == 0,
            dim=-1,
        )
    )

    assert not torch.any(
        torch.all(
            pool.lipid_spectra == 0,
            dim=-1,
        )
    )

    assert (
        pool.n_timepoints
        == target_t
    )

    assert np.isclose(
        pool.bandwidth_hz,
        expected_bandwidth,
    )

    assert (
        pool.device
        == pool.water_spectra.device
    )

    assert (
        pool.lipid_spectra.device
        == pool.device
    )

    assert (
        pool.water_offsets.device
        == pool.device
    )

    assert (
        pool.lipid_offsets.device
        == pool.device
    )

    assert (
        pool.native_lengths.device
        == pool.device
    )

    if (
        simulation_cfg
        .lipid_projection
        .enabled
    ):
        assert (
            pool
            .lipid_projection_operators
            is not None
        )

        assert (
            pool
            .lipid_projection_operators
            .shape
            == (
                pool.n_subjects,
                target_t,
                target_t,
            )
        )

        assert (
            pool
            .lipid_projection_operators
            .dtype
            == torch.complex64
        )

        assert (
            pool
            .lipid_projection_operators
            .device
            == pool.device
        )

        assert torch.isfinite(
            pool
            .lipid_projection_operators
            .real
        ).all()

        assert torch.isfinite(
            pool
            .lipid_projection_operators
            .imag
        ).all()

    else:
        assert (
            pool
            .lipid_projection_operators
            is None
        )

In [ ]:
validate_pool(
    resources.train,
    train_cfg.data.train_subjects,
)

validate_pool(
    resources.validation,
    train_cfg.data.val_subjects,
)


overlap = (
    set(
        resources
        .train
        .subject_names
    )
    & set(
        resources
        .validation
        .subject_names
    )
)

assert not overlap, (
    "Train/validation overlap found:\n"
    f"{sorted(overlap)}"
)

print(
    "All structural pool checks passed."
)

In [ ]:
def subject_table(
    pool: SimulationPool,
) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "subject_index": (
                np.arange(
                    pool.n_subjects
                )
            ),
            "subject": (
                pool.subject_names
            ),
            "water_fids": (
                pool
                .water_counts
                .cpu()
                .numpy()
            ),
            "lipid_fids": (
                pool
                .lipid_counts
                .cpu()
                .numpy()
            ),
            "native_timepoints": (
                pool
                .native_lengths
                .cpu()
                .numpy()
            ),
        }
    )


print("Training subjects")

display(
    subject_table(
        resources.train
    )
)


print("Validation subjects")

display(
    subject_table(
        resources.validation
    )
)

In [ ]:
POOL_TO_INSPECT = (
    resources.train
)

SUBJECT_INDEX = 0


subject_name = (
    POOL_TO_INSPECT
    .subject_name(
        SUBJECT_INDEX
    )
)

water_subject = (
    POOL_TO_INSPECT
    .water_for_subject(
        SUBJECT_INDEX
    )
)

lipid_subject = (
    POOL_TO_INSPECT
    .lipids_for_subject(
        SUBJECT_INDEX
    )
)


print(
    "Subject:",
    subject_name,
)

print(
    "Water subset:",
    tuple(
        water_subject.shape
    ),
)

print(
    "Lipid subset:",
    tuple(
        lipid_subject.shape
    ),
)

In [ ]:
water_start = int(
    POOL_TO_INSPECT
    .water_offsets[
        SUBJECT_INDEX
    ]
)

water_end = int(
    POOL_TO_INSPECT
    .water_offsets[
        SUBJECT_INDEX + 1
    ]
)

lipid_start = int(
    POOL_TO_INSPECT
    .lipid_offsets[
        SUBJECT_INDEX
    ]
)

lipid_end = int(
    POOL_TO_INSPECT
    .lipid_offsets[
        SUBJECT_INDEX + 1
    ]
)


assert torch.equal(
    water_subject,
    POOL_TO_INSPECT
    .water_spectra[
        water_start:water_end
    ],
)

assert torch.equal(
    lipid_subject,
    POOL_TO_INSPECT
    .lipid_spectra[
        lipid_start:lipid_end
    ],
)

print(
    "Subject helper methods agree "
    "with the stored offsets."
)

In [ ]:
WATER_EXAMPLE_INDEX = 0

water_spectrum = (
    water_subject[
        WATER_EXAMPLE_INDEX
    ]
    .detach()
    .cpu()
    .numpy()
)

water_fid = np.fft.ifft(
    np.fft.ifftshift(
        water_spectrum
    )
)


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        water_fid
    )
)

plt.title(
    f"Water FID reconstructed from spectrum – {subject_name}"
)

plt.xlabel(
    "Timepoint"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        water_spectrum
    )
)

plt.title(
    f"Water spectrum – {subject_name}"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
LIPID_EXAMPLE_INDEX = 0

lipid_spectrum = (
    lipid_subject[
        LIPID_EXAMPLE_INDEX
    ]
    .detach()
    .cpu()
    .numpy()
)

lipid_fid = np.fft.ifft(
    np.fft.ifftshift(
        lipid_spectrum
    )
)


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        lipid_fid
    )
)

plt.title(
    f"Lipid FID reconstructed from spectrum – {subject_name}"
)

plt.xlabel(
    "Timepoint"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        lipid_spectrum
    )
)

plt.title(
    f"Lipid spectrum – {subject_name}"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
def sample_global_indices_for_subjects(
    offsets: torch.Tensor,
    subject_indices: torch.Tensor,
    *,
    generator: torch.Generator,
) -> torch.Tensor:
    """
    Wähle für jedes angegebene Subject einen zufälligen
    lokalen FID-Index und rechne ihn in einen globalen
    Pool-Index um.
    """
    counts = (
        offsets[1:]
        - offsets[:-1]
    )

    selected_counts = counts[
        subject_indices
    ]

    random_values = torch.rand(
        subject_indices.shape,
        generator=generator,
        device=subject_indices.device,
    )

    local_indices = torch.floor(
        random_values
        * selected_counts
    ).to(
        torch.int64
    )

    global_indices = (
        offsets[
            subject_indices
        ]
        + local_indices
    )

    return global_indices

In [ ]:
test_generator_1 = (
    torch.Generator(
        device="cpu"
    )
)

test_generator_1.manual_seed(
    12345
)


test_generator_2 = (
    torch.Generator(
        device="cpu"
    )
)

test_generator_2.manual_seed(
    12345
)


subject_indices = torch.tensor(
    [
        0,
        0,
        1,
        1,
        resources.train.n_subjects - 1,
    ],
    dtype=torch.int64,
)


indices_1 = (
    sample_global_indices_for_subjects(
        resources
        .train
        .water_offsets,
        subject_indices,
        generator=test_generator_1,
    )
)

indices_2 = (
    sample_global_indices_for_subjects(
        resources
        .train
        .water_offsets,
        subject_indices,
        generator=test_generator_2,
    )
)


print(
    "Subject indices:",
    subject_indices.tolist(),
)

print(
    "Sampled global water indices:",
    indices_1.tolist(),
)


assert torch.equal(
    indices_1,
    indices_2,
)


for (
    subject_index,
    global_index,
) in zip(
    subject_indices.tolist(),
    indices_1.tolist(),
):
    start = int(
        resources
        .train
        .water_offsets[
            subject_index
        ]
    )

    end = int(
        resources
        .train
        .water_offsets[
            subject_index + 1
        ]
    )

    assert (
        start
        <= global_index
        < end
    )


print(
    "Deterministic subject-aware "
    "index sampling works."
)

In [ ]:
operators = (
    resources
    .train
    .lipid_projection_operators
)


if operators is None:
    print(
        "No projection operators were loaded."
    )

    print(
        "This is expected when "
        "lipid_projection.enabled is false."
    )

else:
    print(
        "Projection operators:",
        tuple(
            operators.shape
        ),
        operators.dtype,
    )

    operator = (
        operators[
            SUBJECT_INDEX
        ]
        .cpu()
        .numpy()
    )

    print(
        "Selected operator shape:",
        operator.shape,
    )

    print(
        "Finite:",
        np.isfinite(
            operator
        ).all(),
    )

In [ ]:
from walinet.training_data.simulator import (
    SampledResources,
    SimulationResourceSampler,
)


sampler = SimulationResourceSampler(
    pool=resources.train,
    config=simulation_cfg,
)

print(
    "Sampler created successfully."
)

print(
    "Mixing mode:",
    sampler.mixing,
)

print(
    "Number of lipid spectra per simulated spectrum:",
    sampler.n_random_lipid_spectra,
)

print(
    "Device:",
    sampler.pool.device,
)

In [ ]:
generator = torch.Generator(
    device=resources.train.device
)

generator.manual_seed(
    12345
)


sampled = sampler.sample(
    batch_size=8,
    generator=generator,
)


print(
    "water_spectra:",
    sampled.water_spectra.shape,
    sampled.water_spectra.dtype,
)

print(
    "lipid_spectra:",
    sampled.lipid_spectra.shape,
    sampled.lipid_spectra.dtype,
)

print(
    "water_subject_indices:",
    sampled.water_subject_indices,
)

print(
    "lipid_subject_indices:",
    sampled.lipid_subject_indices,
)

print(
    "water_resource_indices:",
    sampled.water_resource_indices,
)

print(
    "lipid_resource_indices:",
    sampled.lipid_resource_indices,
)

In [ ]:
water_subject_names = [
    resources.train.subject_names[
        subject_index
    ]
    for subject_index in (
        sampled
        .water_subject_indices
        .cpu()
        .tolist()
    )
]

lipid_subject_names = [
    resources.train.subject_names[
        subject_index
    ]
    for subject_index in (
        sampled
        .lipid_subject_indices
        .cpu()
        .tolist()
    )
]


for batch_index, (
    water_subject,
    lipid_subject,
) in enumerate(
    zip(
        water_subject_names,
        lipid_subject_names,
    )
):
    print(
        f"{batch_index}: "
        f"water={water_subject}, "
        f"lipids={lipid_subject}"
    )

In [ ]:
generator_1 = torch.Generator(
    device=resources.train.device
)

generator_1.manual_seed(
    12345
)


generator_2 = torch.Generator(
    device=resources.train.device
)

generator_2.manual_seed(
    12345
)


sampled_1 = sampler.sample(
    batch_size=32,
    generator=generator_1,
)

sampled_2 = sampler.sample(
    batch_size=32,
    generator=generator_2,
)


assert torch.equal(
    sampled_1.water_subject_indices,
    sampled_2.water_subject_indices,
)

assert torch.equal(
    sampled_1.lipid_subject_indices,
    sampled_2.lipid_subject_indices,
)

assert torch.equal(
    sampled_1.water_resource_indices,
    sampled_2.water_resource_indices,
)

assert torch.equal(
    sampled_1.lipid_resource_indices,
    sampled_2.lipid_resource_indices,
)

assert torch.equal(
    sampled_1.water_spectra,
    sampled_2.water_spectra,
)

assert torch.equal(
    sampled_1.lipid_spectra,
    sampled_2.lipid_spectra,
)


print(
    "Sampling is fully deterministic "
    "for identical seeds."
)

In [ ]:
for batch_index in range(
    sampled.batch_size
):
    water_subject_index = int(
        sampled
        .water_subject_indices[
            batch_index
        ]
    )

    water_resource_index = int(
        sampled
        .water_resource_indices[
            batch_index
        ]
    )

    water_start = int(
        resources
        .train
        .water_offsets[
            water_subject_index
        ]
    )

    water_end = int(
        resources
        .train
        .water_offsets[
            water_subject_index + 1
        ]
    )

    assert (
        water_start
        <= water_resource_index
        < water_end
    )


    lipid_subject_index = int(
        sampled
        .lipid_subject_indices[
            batch_index
        ]
    )

    lipid_start = int(
        resources
        .train
        .lipid_offsets[
            lipid_subject_index
        ]
    )

    lipid_end = int(
        resources
        .train
        .lipid_offsets[
            lipid_subject_index + 1
        ]
    )

    lipid_indices = (
        sampled
        .lipid_resource_indices[
            batch_index
        ]
    )

    assert torch.all(
        lipid_indices
        >= lipid_start
    )

    assert torch.all(
        lipid_indices
        < lipid_end
    )


print(
    "All sampled spectra belong to "
    "their selected subjects."
)

In [ ]:
batch_index = 0

water_spectrum = (
    sampled
    .water_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

lipid_spectra = (
    sampled
    .lipid_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)


# Optional: reconstruct FIDs for inspection.
water_fid = np.fft.ifft(
    np.fft.ifftshift(
        water_spectrum
    )
)

lipid_fids = np.fft.ifft(
    np.fft.ifftshift(
        lipid_spectra,
        axes=-1,
    ),
    axis=-1,
)


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        water_fid
    )
)

plt.title(
    "Sampled water FID reconstructed from spectrum"
)

plt.xlabel(
    "Timepoint"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


plt.figure(
    figsize=(12, 5)
)

for lipid_index in range(
    lipid_fids.shape[0]
):
    plt.plot(
        np.abs(
            lipid_fids[
                lipid_index
            ]
        ),
        alpha=0.7,
    )

plt.title(
    "Sampled lipid FIDs reconstructed from spectra"
)

plt.xlabel(
    "Timepoint"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
from walinet.training_data.simulator import (
    LipidMixture,
    mix_sampled_lipid_spectra,
)

In [ ]:
lipid_generator = torch.Generator(
    device=sampled.device
)

lipid_generator.manual_seed(
    23456
)

lipid_mixture = mix_sampled_lipid_spectra(
    sampled=sampled,
    generator=lipid_generator,
)

print(
    "mixed_spectra:",
    lipid_mixture.mixed_spectra.shape,
    lipid_mixture.mixed_spectra.dtype,
)

print(
    "weights:",
    lipid_mixture.weights.shape,
    lipid_mixture.weights.dtype,
)

print(
    "weight sums:",
    lipid_mixture.weights.sum(dim=1),
)

In [ ]:
spectrum_mixed_manually = torch.sum(
    sampled.lipid_spectra
    * lipid_mixture.weights.unsqueeze(-1),
    dim=1,
)

max_difference = torch.max(
    torch.abs(
        spectrum_mixed_manually
        - lipid_mixture.mixed_spectra
    )
)

print(
    "Maximum difference:",
    max_difference.item(),
)

assert torch.allclose(
    spectrum_mixed_manually,
    lipid_mixture.mixed_spectra,
    rtol=1e-6,
    atol=1e-6,
)

print(
    "Spectrum-domain lipid mixing is correct."
)

In [ ]:
batch_index = 0

individual_lipid_spectra = (
    sampled
    .lipid_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

mixed_lipid_spectrum = (
    lipid_mixture
    .mixed_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

plt.figure(
    figsize=(12, 5)
)

for component_index in range(
    individual_lipid_spectra.shape[0]
):
    plt.plot(
        np.abs(
            individual_lipid_spectra[
                component_index
            ]
        ),
        alpha=0.3,
    )

plt.plot(
    np.abs(
        mixed_lipid_spectrum
    ),
    linewidth=2.0,
    label="Weighted mixture",
)

plt.title(
    "Individual lipid spectra and mixed baseline"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
from walinet.training_data.metabolite_simulation import (
    SimulatedMetabolites,
)


prepared_basis = system.prepared_basis

metabolite_simulator = (
    system.metabolite_simulator
)


print(
    "Metabolite simulator created."
)

print(
    "Basis library:",
    simulation_cfg
    .basis
    .library,
)

print(
    "Basis components:",
    metabolite_simulator
    .n_basis_components,
)

print(
    "Metabolite profiles:",
    len(
        metabolite_simulator
        .sampling_tables
    ),
)

for index, sampling_table in enumerate(
    metabolite_simulator
    .sampling_tables
):
    profile_cfg = (
        simulation_cfg
        .metabolites
        .profiles[index]
    )

    print(
        f"  Profile [{index}]:"
    )

    print(
        "    Config:",
        profile_cfg.config,
    )

    print(
        "    Probability:",
        profile_cfg.probability,
    )

    print(
        "    Active metabolites:",
        sampling_table
        .n_active_components,
    )

print(
    "Timepoints:",
    metabolite_simulator
    .n_timepoints,
)

print(
    "Device:",
    metabolite_simulator
    .device,
)

In [ ]:
print("Active metabolite mappings")
print("=" * 60)

for profile_index, table in enumerate(
    metabolite_simulator
    .sampling_tables
):
    profile_cfg = (
        simulation_cfg
        .metabolites
        .profiles[profile_index]
    )

    print()
    print(
        f"Profile [{profile_index}]"
    )

    print(
        "Config:",
        profile_cfg.config,
    )

    print(
        "Probability:",
        profile_cfg.probability,
    )

    print("-" * 60)

    for (
        config_name,
        basis_name,
    ) in zip(
        table.active_config_names,
        table.active_basis_names,
    ):
        basis_index = (
            table
            .basis_names
            .index(
                basis_name
            )
        )

        print(
            f"{config_name:8s} "
            f"-> {basis_name:12s} "
            f"mean={table.means[basis_index].item():6.2f} "
            f"std={table.stds[basis_index].item():6.2f}"
        )

In [ ]:
metabolite_generator = torch.Generator(
    device=system.device
)

metabolite_generator.manual_seed(
    34567
)


simulated_metabolites = (
    metabolite_simulator.simulate(
        batch_size=8,
        generator=metabolite_generator,
    )
)


print(
    "clean_fids:",
    simulated_metabolites.clean_fids.shape,
    simulated_metabolites.clean_fids.dtype,
)

print(
    "clean_spectra:",
    simulated_metabolites.clean_spectra.shape,
    simulated_metabolites.clean_spectra.dtype,
)

print(
    "concentrations:",
    simulated_metabolites.concentrations.shape,
    simulated_metabolites.concentrations.dtype,
)

print(
    "selected profiles:",
    simulated_metabolites.profile_indices,
)

print(
    "acquisition delays [s]:",
    simulated_metabolites
    .acquisition_delays_seconds,
)

print(
    "frequency shifts [Hz]:",
    simulated_metabolites
    .frequency_shifts_hz,
)

print(
    "global phases [rad]:",
    simulated_metabolites
    .global_phases_radians,
)

In [ ]:
cfg_metab = simulation_cfg.metabolites

# ---------------------------------------------------------
# Acquisition delay
# ---------------------------------------------------------
assert torch.all(
    torch.abs(
        simulated_metabolites
        .acquisition_delays_seconds
    )
    <= (
        cfg_metab
        .max_acquisition_delay_seconds
        + 1e-8
    )
)

# ---------------------------------------------------------
# Frequency shift
# Normal verteilt, daher kein fester Min-/Max-Bereich.
# Hier nur Plausibilitätschecks.
# ---------------------------------------------------------
assert torch.isfinite(
    simulated_metabolites
    .frequency_shifts_hz
).all()

# ---------------------------------------------------------
# Voigt FWHM
# Positiv normalverteilt durch Rejection Sampling.
# ---------------------------------------------------------
assert torch.all(
    simulated_metabolites
    .voigt_fwhm_hz
    > 0
)

# ---------------------------------------------------------
# Lorentzian fraction
# Intern uniform in [0, 1].
# ---------------------------------------------------------
assert torch.all(
    simulated_metabolites
    .lorentzian_fractions
    >= 0
)

assert torch.all(
    simulated_metabolites
    .lorentzian_fractions
    <= 1
)

# ---------------------------------------------------------
# Gaussian and Lorentzian FWHM
# ---------------------------------------------------------
assert torch.all(
    simulated_metabolites
    .gaussian_fwhm_hz
    >= 0
)

assert torch.all(
    simulated_metabolites
    .lorentzian_fwhm_hz
    >= 0
)

# ---------------------------------------------------------
# Reconstruct total Voigt FWHM
# ---------------------------------------------------------
a = 0.5346
b = 0.2166

reconstructed_voigt_fwhm = (
    a
    * simulated_metabolites
    .lorentzian_fwhm_hz
    + torch.sqrt(
        b
        * simulated_metabolites
        .lorentzian_fwhm_hz.square()
        + simulated_metabolites
        .gaussian_fwhm_hz.square()
    )
)

assert torch.allclose(
    reconstructed_voigt_fwhm,
    simulated_metabolites
    .voigt_fwhm_hz,
    rtol=1e-5,
    atol=1e-5,
)

# ---------------------------------------------------------
# General finite-value checks
# ---------------------------------------------------------
for name, values in {
    "acquisition_delays_seconds": (
        simulated_metabolites
        .acquisition_delays_seconds
    ),
    "frequency_shifts_hz": (
        simulated_metabolites
        .frequency_shifts_hz
    ),
    "voigt_fwhm_hz": (
        simulated_metabolites
        .voigt_fwhm_hz
    ),
    "lorentzian_fractions": (
        simulated_metabolites
        .lorentzian_fractions
    ),
    "gaussian_fwhm_hz": (
        simulated_metabolites
        .gaussian_fwhm_hz
    ),
    "lorentzian_fwhm_hz": (
        simulated_metabolites
        .lorentzian_fwhm_hz
    ),
}.items():
    assert torch.isfinite(
        values
    ).all(), (
        f"{name} contains NaN or Inf."
    )

print(
    "All sampled metabolite parameters are valid."
)

In [ ]:
for profile_index, table in enumerate(
    metabolite_simulator.sampling_tables
):
    selected_rows = (
        simulated_metabolites.profile_indices
        == profile_index
    )

    if not torch.any(selected_rows):
        continue

    disabled_mask = ~table.enabled_mask

    disabled_concentrations = (
        simulated_metabolites
        .concentrations[
            selected_rows
        ][
            :,
            disabled_mask,
        ]
    )

    assert torch.all(
        disabled_concentrations == 0
    ), (
        f"Disabled basis components are non-zero "
        f"for profile {profile_index}."
    )

print(
    "All disabled basis components have "
    "zero concentration in their selected profile."
)

In [ ]:
generator_1 = torch.Generator(
    device=system.device
)
generator_1.manual_seed(
    34567
)

generator_2 = torch.Generator(
    device=system.device
)
generator_2.manual_seed(
    34567
)


metabolites_1 = (
    metabolite_simulator.simulate(
        batch_size=32,
        generator=generator_1,
    )
)

metabolites_2 = (
    metabolite_simulator.simulate(
        batch_size=32,
        generator=generator_2,
    )
)


assert torch.equal(
    metabolites_1.profile_indices,
    metabolites_2.profile_indices,
)

assert torch.equal(
    metabolites_1.concentrations,
    metabolites_2.concentrations,
)

assert torch.equal(
    metabolites_1.clean_fids,
    metabolites_2.clean_fids,
)

assert torch.equal(
    metabolites_1.clean_spectra,
    metabolites_2.clean_spectra,
)

assert torch.equal(
    metabolites_1.acquisition_delays_seconds,
    metabolites_2.acquisition_delays_seconds,
)

assert torch.equal(
    metabolites_1.global_phases_radians,
    metabolites_2.global_phases_radians,
)

assert torch.equal(
    metabolites_1.frequency_shifts_hz,
    metabolites_2.frequency_shifts_hz,
)

assert torch.equal(
    metabolites_1.voigt_fwhm_hz,
    metabolites_2.voigt_fwhm_hz,
)

assert torch.equal(
    metabolites_1.lorentzian_fractions,
    metabolites_2.lorentzian_fractions,
)

assert torch.equal(
    metabolites_1.gaussian_fwhm_hz,
    metabolites_2.gaussian_fwhm_hz,
)

assert torch.equal(
    metabolites_1.lorentzian_fwhm_hz,
    metabolites_2.lorentzian_fwhm_hz,
)

print(
    "Metabolite simulation is deterministic "
    "for identical seeds."
)

In [ ]:
spectra = (
    simulated_metabolites
    .clean_spectra
    .cpu()
    .numpy()
)

plt.figure(
    figsize=(12, 6)
)

for index in range(
    spectra.shape[0]
):
    plt.plot(
        np.real(
            spectra[index]
        ),
        alpha=0.7,
    )

plt.title(
    "Simulated noise-free metabolite spectra"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Real signal"
)

plt.show()

In [ ]:
from walinet.training_data.metabolite_noise import (
    SimulatedNoise,
    simulate_receiver_noise,
)


noise_generator = torch.Generator(
    device=simulated_metabolites.device
)

noise_generator.manual_seed(
    45678
)


simulated_noise = simulate_receiver_noise(
    metabolites=simulated_metabolites,
    config=simulation_cfg,
    generator=noise_generator,
)


print(
    "noise_spectra:",
    simulated_noise.noise_spectra.shape,
    simulated_noise.noise_spectra.dtype,
)

print(
    "SNR:",
    simulated_noise.snr,
)

print(
    "clean_spectrum_std:",
    simulated_noise.clean_spectrum_std,
)

print(
    "noise_scale:",
    simulated_noise.noise_scale,
)

In [ ]:
expected_noisy_metabolite_spectra = (
    simulated_metabolites.clean_spectra
    + simulated_noise.noise_spectra
)

assert (
    simulated_noise.noise_spectra.shape
    == simulated_metabolites.clean_spectra.shape
)

assert (
    simulated_noise.noise_spectra.dtype
    == simulated_metabolites.clean_spectra.dtype
)

assert (
    simulated_noise.noise_spectra.device
    == simulated_metabolites.clean_spectra.device
)

assert torch.all(
    simulated_noise.snr
    >= simulation_cfg.noise.snr_min
)

assert torch.all(
    simulated_noise.snr
    <= simulation_cfg.noise.snr_max
)

assert torch.all(
    simulated_noise.noise_scale
    > 0
)

assert torch.isfinite(
    simulated_noise.noise_spectra.real
).all()

assert torch.isfinite(
    simulated_noise.noise_spectra.imag
).all()

assert torch.isfinite(
    expected_noisy_metabolite_spectra.real
).all()

assert torch.isfinite(
    expected_noisy_metabolite_spectra.imag
).all()


# Verify the implemented scaling rule:
#
# noise_scale =
#     std(clean spectrum)
#     / 0.65
#     / SNR

expected_noise_scale = (
    simulated_noise.clean_spectrum_std
    / 0.65
    / simulated_noise.snr
)

assert torch.allclose(
    simulated_noise.noise_scale,
    expected_noise_scale,
    rtol=1e-6,
    atol=1e-6,
)


print(
    "Frequency-domain receiver-noise simulation "
    "is consistent."
)

In [ ]:
batch_index = 0

clean_spectrum = (
    simulated_metabolites
    .clean_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

noise_spectrum = (
    simulated_noise
    .noise_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

noisy_spectrum = (
    clean_spectrum
    + noise_spectrum
)


plt.figure(
    figsize=(12, 5)
)

plt.plot(
    np.real(clean_spectrum),
    label="Clean metabolites",
)

plt.plot(
    np.real(noisy_spectrum),
    label="Metabolites + receiver noise",
    alpha=0.8,
)

plt.title(
    "Frequency-domain receiver noise"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Real signal"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.real(
        noise_spectrum
    )
)

plt.title(
    f"Noise spectrum, sampled SNR = "
    f"{simulated_noise.snr[batch_index].item():.2f}"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Real noise"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
from walinet.training_data.metabolite_noise import (
    simulate_receiver_noise,
)
from walinet.training_data.simulator import (
    mix_sampled_lipid_spectra,
)
from walinet.training_data.spectrum_assembly import (
    AssembledSpectra,
    assemble_spectra,
)


device = system.device
pool = system.resources.train
batch_size = 8


# ---------------------------------------------------------
# Sample water and lipid resources
# ---------------------------------------------------------
resource_generator = torch.Generator(
    device=device,
)

resource_generator.manual_seed(
    12345,
)

sampled = (
    system.train_simulator
    .resource_sampler
    .sample(
        batch_size=batch_size,
        generator=resource_generator,
    )
)


# ---------------------------------------------------------
# Mix sampled lipid spectra
# ---------------------------------------------------------
lipid_generator = torch.Generator(
    device=device,
)

lipid_generator.manual_seed(
    23456,
)

lipid_mixture = mix_sampled_lipid_spectra(
    sampled=sampled,
    generator=lipid_generator,
)


# ---------------------------------------------------------
# Simulate metabolites
# ---------------------------------------------------------
metabolite_generator = torch.Generator(
    device=device,
)

metabolite_generator.manual_seed(
    34567,
)

simulated_metabolites = (
    system.metabolite_simulator.simulate(
        batch_size=batch_size,
        generator=metabolite_generator,
    )
)


# ---------------------------------------------------------
# Simulate receiver noise
# ---------------------------------------------------------
noise_generator = torch.Generator(
    device=device,
)

noise_generator.manual_seed(
    45678,
)

simulated_noise = simulate_receiver_noise(
    metabolites=simulated_metabolites,
    config=system.simulation_config,
    generator=noise_generator,
)


# ---------------------------------------------------------
# Assemble complete spectra
# ---------------------------------------------------------
assembly_generator = torch.Generator(
    device=device,
)

assembly_generator.manual_seed(
    56789,
)

assembled = assemble_spectra(
    sampled=sampled,
    lipid_mixture=lipid_mixture,
    metabolites=simulated_metabolites,
    noise=simulated_noise,
    pool=pool,
    config=system.simulation_config,
    generator=assembly_generator,
)


# ---------------------------------------------------------
# Output
# ---------------------------------------------------------
print(
    "Sampled device:",
    sampled.device,
)

print(
    "Lipid mixture device:",
    lipid_mixture.device,
)

print(
    "Metabolites device:",
    simulated_metabolites.device,
)

print(
    "Noise device:",
    simulated_noise.device,
)

print(
    "Pool device:",
    pool.device,
)

print()

print(
    "Clean metabolites:",
    assembled.clean_metabolite_spectra.shape,
)

print(
    "Baseline target:",
    assembled.baseline_spectra.shape,
)

print(
    "Target property:",
    assembled.target_spectra.shape,
)

print(
    "Water:",
    assembled.water_spectra.shape,
)

print(
    "Lipids:",
    assembled.lipid_spectra.shape,
)

print(
    "Clean mixture:",
    assembled.clean_mixture_spectra.shape,
)

print(
    "Final noisy mixture:",
    assembled.mixture_spectra.shape,
)

print(
    "Projected:",
    (
        None
        if assembled.projected_spectra is None
        else assembled.projected_spectra.shape
    ),
)

print(
    "Final network input:",
    assembled.input_spectra.shape,
)

print(
    "Acquired lengths:",
    assembled.acquired_n_timepoints.min().item(),
    "to",
    assembled.acquired_n_timepoints.max().item(),
)

print(
    "Water scaling:",
    assembled.water_scaling,
)

print(
    "Lipid scaling:",
    assembled.lipid_scaling,
)

print(
    "SNR:",
    assembled.noise.snr,
)

print(
    "Assembly device:",
    assembled.input_spectra.device,
)

In [ ]:
batch_index = 0

metabolites_np = (
    assembled
    .metabolite_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

water_np = (
    assembled
    .water_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

lipids_np = (
    assembled
    .lipid_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

mixture_np = (
    assembled
    .mixture_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

In [ ]:
spectrum_index = 0

metabolites_np = (
    assembled
    .clean_metabolite_spectra[spectrum_index]
    .detach()
    .cpu()
    .numpy()
)

water_np = (
    assembled
    .water_spectra[spectrum_index]
    .detach()
    .cpu()
    .numpy()
)

lipids_np = (
    assembled
    .lipid_spectra[spectrum_index]
    .detach()
    .cpu()
    .numpy()
)


frequency_axis_hz = np.fft.fftshift(
    np.fft.fftfreq(
        simulation_cfg.acquisition.n_timepoints,
        d=(
            1.0
            / simulation_cfg
            .acquisition
            .bandwidth_hz
        ),
    )
)

ppm_axis = (
    prepared_basis.ppm_reference
    - frequency_axis_hz
    / prepared_basis.hz_per_ppm
)


plt.figure(
    figsize=(12, 5)
)

plt.plot(
    ppm_axis,
    np.real(metabolites_np),
    label="Metabolites",
)

plt.plot(
    ppm_axis,
    np.real(water_np),
    label="Water",
)

plt.plot(
    ppm_axis,
    np.real(lipids_np),
    label="Lipids",
)

plt.xlim(
    7.5,
    0.0,
)

plt.xlabel(
    "Chemical shift [ppm]"
)

plt.ylabel(
    "Real signal"
)

plt.title(
    "Simulated spectral components"
)

plt.legend()
plt.grid(
    alpha=0.3
)
plt.tight_layout()
plt.show()

In [ ]:
print("metabolites_np shape:", metabolites_np.shape)
print("lipids_np shape:", lipids_np.shape)

print(
    "Max |metabolites_np|:",
    np.max(np.abs(metabolites_np)),
)

print(
    "Max |clean metabolites|:",
    np.max(
        np.abs(
            assembled
            .clean_metabolite_spectra[0]
            .detach()
            .cpu()
            .numpy()
        )
    ),
)

print(
    "Max |water|:",
    np.max(
        np.abs(
            assembled
            .water_spectra[0]
            .detach()
            .cpu()
            .numpy()
        )
    ),
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    ppm_axis,
    np.real(water_np),
    label="Water",
)

plt.plot(
    ppm_axis,
    np.real(mixture_np),
    label="Complete mixture",
    alpha=0.8,
)

plt.xlim(3.5, 0.0)
plt.xlabel("Chemical shift [ppm]")
plt.ylabel("Real signal")
plt.title("Simulated water and complete spectrum")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch


sorted_indices = torch.argsort(
    simulated_noise.snr
)

positions = np.linspace(
    0,
    len(sorted_indices) - 1,
    num=min(4, len(sorted_indices)),
    dtype=int,
)

selected_indices = [
    int(sorted_indices[position].item())
    for position in positions
]


visible_mask = (
    (ppm_axis >= 0.0)
    & (ppm_axis <= 7.5)
)


for batch_index in selected_indices:
    water_np = (
        assembled.water_spectra[
            batch_index
        ]
        .detach()
        .cpu()
        .numpy()
    )

    mixture_np = (
        assembled.input_spectra[
            batch_index
        ]
        .detach()
        .cpu()
        .numpy()
    )

    snr_value = float(
        simulated_noise.snr[
            batch_index
        ].item()
    )

    plt.figure(
        figsize=(12, 5)
    )

    plt.plot(
        ppm_axis[visible_mask],
        np.real(
            water_np[visible_mask]
        ),
        label="Water",
    )

    plt.plot(
        ppm_axis[visible_mask],
        np.real(
            mixture_np[visible_mask]
        ),
        label="Network input",
        alpha=0.8,
    )

    plt.xlim(
        7.5,
        0.0,
    )

    plt.xlabel(
        "Chemical shift [ppm]"
    )

    plt.ylabel(
        "Real signal"
    )

    plt.title(
        f"Simulated spectrum {batch_index}, "
        f"SNR = {snr_value:.2f}"
    )

    plt.legend()

    plt.grid(
        alpha=0.3
    )

    plt.tight_layout()

    plt.show()

In [ ]:
from walinet.training_data.simulator import (
    SimulationResourceSampler,
    mix_sampled_lipid_spectra,
)


resource_sampler = SimulationResourceSampler(
    pool=resources.train,
    config=simulation_cfg,
)


print(
    "Resource sampler created."
)

print(
    "Device:",
    resource_sampler.pool.device,
)

print(
    "Timepoints:",
    resource_sampler.pool.n_timepoints,
)

In [ ]:
batch_size = 32

generator = torch.Generator(
    device=system.device,
)
generator.manual_seed(
    12345,
)


sampled_resources = (
    system.train_simulator
    .resource_sampler
    .sample(
        batch_size=batch_size,
        generator=generator,
    )
)

lipid_mixture = mix_sampled_lipid_spectra(
    sampled=sampled_resources,
    generator=generator,
)

simulated_metabolites = (
    system.metabolite_simulator
    .simulate(
        batch_size=batch_size,
        generator=generator,
    )
)

simulated_noise = simulate_receiver_noise(
    metabolites=simulated_metabolites,
    config=system.simulation_config,
    generator=generator,
)

assembled = assemble_spectra(
    sampled=sampled_resources,
    lipid_mixture=lipid_mixture,
    metabolites=simulated_metabolites,
    noise=simulated_noise,
    pool=system.resources.train,
    config=system.simulation_config,
    generator=generator,
)

In [ ]:
expected_shape = (
    batch_size,
    system.simulation_config
    .acquisition
    .n_timepoints,
)

expected_device = system.device


print(
    "Input:",
    assembled.input_spectra.shape,
    assembled.input_spectra.dtype,
    assembled.input_spectra.device,
)

print(
    "Target:",
    assembled.target_spectra.shape,
    assembled.target_spectra.dtype,
    assembled.target_spectra.device,
)

print(
    "Acquired lengths:",
    assembled.acquired_n_timepoints.min().item(),
    "to",
    assembled.acquired_n_timepoints.max().item(),
)

print(
    "SNR:",
    assembled.noise.snr.min().item(),
    "to",
    assembled.noise.snr.max().item(),
)


assert tuple(
    assembled.input_spectra.shape
) == expected_shape

assert tuple(
    assembled.target_spectra.shape
) == expected_shape

assert (
    assembled.input_spectra.device
    == expected_device
)

assert (
    assembled.target_spectra.device
    == expected_device
)

assert torch.is_complex(
    assembled.input_spectra
)

assert torch.is_complex(
    assembled.target_spectra
)

assert torch.all(
    assembled.acquired_n_timepoints
    >= system.simulation_config
    .acquisition
    .min_acquired_n_timepoints
)

assert torch.all(
    assembled.acquired_n_timepoints
    <= system.simulation_config
    .acquisition
    .max_acquired_n_timepoints
)

assert torch.all(
    assembled.noise.snr
    >= system.simulation_config
    .noise
    .snr_min
)

assert torch.all(
    assembled.noise.snr
    <= system.simulation_config
    .noise
    .snr_max
)

print(
    "Basic end-to-end checks passed."
)

In [ ]:
def assert_finite_complex(
    tensor: torch.Tensor,
    name: str,
) -> None:
    assert torch.isfinite(
        tensor.real
    ).all(), f"{name} contains non-finite real values."

    assert torch.isfinite(
        tensor.imag
    ).all(), f"{name} contains non-finite imaginary values."


assert_finite_complex(
    assembled.input_spectra,
    "input_spectra",
)

assert_finite_complex(
    assembled.metabolite_spectra,
    "metabolite_spectra",
)

assert_finite_complex(
    assembled.clean_mixture_spectra,
    "clean_mixture_spectra",
)

assert_finite_complex(
    assembled.water_spectra,
    "water_spectra",
)

assert_finite_complex(
    assembled.lipid_spectra,
    "lipid_spectra",
)

print(
    "All spectra are finite."
)

In [ ]:
input_fids = torch.fft.ifft(
    torch.fft.ifftshift(
        assembled.mixture_spectra,
        dim=-1,
    ),
    dim=-1,
)

target_fids = torch.fft.ifft(
    torch.fft.ifftshift(
        assembled.target_spectra,
        dim=-1,
    ),
    dim=-1,
)


time_indices = torch.arange(
    assembled.n_timepoints,
    device=assembled.device,
)

tail_mask = (
    time_indices[None, :]
    >= assembled.acquired_n_timepoints[:, None]
)


if torch.any(tail_mask):
    input_tail_max = (
        torch.abs(input_fids)
        .masked_select(tail_mask)
        .max()
    )

    target_tail_max = (
        torch.abs(target_fids)
        .masked_select(tail_mask)
        .max()
    )
else:
    input_tail_max = torch.tensor(
        0.0,
        device=assembled.device,
    )

    target_tail_max = torch.tensor(
        0.0,
        device=assembled.device,
    )


input_reference = torch.abs(
    input_fids
).max()

target_reference = torch.abs(
    target_fids
).max()


input_relative_tail = (
    input_tail_max
    / input_reference.clamp_min(
        torch.finfo(
            input_reference.dtype
        ).tiny
    )
)

target_relative_tail = (
    target_tail_max
    / target_reference.clamp_min(
        torch.finfo(
            target_reference.dtype
        ).tiny
    )
)


print(
    "Projection enabled:",
    simulation_cfg.lipid_projection.enabled,
)

print(
    "Input tail absolute maximum:",
    input_tail_max.item(),
)

print(
    "Input tail relative maximum:",
    input_relative_tail.item(),
)

print(
    "Target tail absolute maximum:",
    target_tail_max.item(),
)

print(
    "Target tail relative maximum:",
    target_relative_tail.item(),
)


assert input_relative_tail < 1e-5
assert target_relative_tail < 1e-5

print(
    "Zero-filled FID tails are correct."
)

In [ ]:
from dataclasses import replace

from walinet.training_data.acquisition_length import (
    simulate_acquisition_length,
)


full_length_config = replace(
    simulation_cfg,
    acquisition=replace(
        simulation_cfg.acquisition,
        min_acquired_n_timepoints=(
            simulation_cfg
            .acquisition
            .n_timepoints
        ),
        max_acquired_n_timepoints=(
            simulation_cfg
            .acquisition
            .n_timepoints
        ),
    ),
)


test_spectra = torch.stack(
    (
        assembled.clean_mixture_spectra,
        assembled.clean_metabolite_spectra,
    ),
    dim=1,
)


fast_path_generator = torch.Generator(
    device=test_spectra.device,
)

fast_path_generator.manual_seed(
    12345
)


fast_path_result = (
    simulate_acquisition_length(
        spectra=test_spectra,
        config=full_length_config,
        generator=fast_path_generator,
    )
)


assert torch.equal(
    fast_path_result.spectra,
    test_spectra,
)

assert (
    fast_path_result.spectra.data_ptr()
    == test_spectra.data_ptr()
)

assert torch.all(
    fast_path_result.acquired_n_timepoints
    == simulation_cfg.acquisition.n_timepoints
)

print(
    "Full-length fast path returns the original tensor "
    "without copying or FFTs."
)

In [ ]:
from walinet.training_data.spectrum_simulator import (
    SimulatedSpectrumBatch,
    SpectrumSimulator,
)


spectrum_simulator = (
    system.train_simulator
)


print(
    "Spectrum simulator created."
)

print(
    "Device:",
    spectrum_simulator.device,
)

print(
    "Timepoints:",
    spectrum_simulator.n_timepoints,
)

In [ ]:
generator = torch.Generator(
    device=spectrum_simulator.device,
)

generator.manual_seed(
    12345
)


batch = spectrum_simulator.simulate(
    batch_size=32,
    generator=generator,
)


print(
    "Raw input:",
    batch.raw_input_spectra.shape,
    batch.raw_input_spectra.dtype,
    batch.raw_input_spectra.device,
)

print(
    "Raw target:",
    batch.raw_target_spectra.shape,
    batch.raw_target_spectra.dtype,
    batch.raw_target_spectra.device,
)

print(
    "Normalized input:",
    batch.normalized_input_spectra.shape,
    batch.normalized_input_spectra.dtype,
    batch.normalized_input_spectra.device,
)

print(
    "Normalized target:",
    batch.normalized_target_spectra.shape,
    batch.normalized_target_spectra.dtype,
    batch.normalized_target_spectra.device,
)

print(
    "Network input:",
    batch.network_input.shape,
    batch.network_input.dtype,
    batch.network_input.device,
)

print(
    "Network target:",
    batch.network_target.shape,
    batch.network_target.dtype,
    batch.network_target.device,
)

print(
    "Acquired lengths:",
    batch.acquired_n_timepoints.min().item(),
    "to",
    batch.acquired_n_timepoints.max().item(),
)

print(
    "SNR:",
    batch.snr.min().item(),
    "to",
    batch.snr.max().item(),
)

print(
    "Retries used:",
    batch.retries_used,
)

In [ ]:
import torch

from walinet.training_data.metabolite_simulation import (
    MetaboliteSimulator,
)
from walinet.training_data.spectrum_simulator import (
    SpectrumSimulator,
)


device = torch.device(
    "cuda:0"
)

assert torch.cuda.is_available()


# One-time CPU -> GPU transfer.
gpu_resources = resources.to(
    device
)


# Recreate the metabolite simulator so that basis,
# sampling tables, and constant axes are on the GPU.
gpu_metabolite_simulator = MetaboliteSimulator(
    prepared_basis=prepared_basis,
    config=simulation_cfg,
    device=device,
)


gpu_spectrum_simulator = SpectrumSimulator(
    pool=gpu_resources.train,
    metabolite_simulator=(
        gpu_metabolite_simulator
    ),
    config=simulation_cfg,
)


print(
    "GPU:",
    torch.cuda.get_device_name(
        device
    ),
)

print(
    "Pool device:",
    gpu_resources.train.device,
)

print(
    "Basis device:",
    gpu_metabolite_simulator.basis_fids.device,
)

print(
    "Simulator device:",
    gpu_spectrum_simulator.device,
)


assert gpu_resources.train.device.type == "cuda"
assert gpu_metabolite_simulator.device.type == "cuda"
assert gpu_spectrum_simulator.device.type == "cuda"

In [ ]:
import time
import torch


def benchmark_spectrum_simulator(
    *,
    simulator: SpectrumSimulator,
    total_spectra: int = 100_000,
    batch_size: int = 4096,
    warmup_batches: int = 5,
    seed: int = 123456,
) -> dict[str, float]:
    if simulator.device.type != "cuda":
        raise ValueError(
            "The benchmark requires a CUDA simulator, "
            f"but found {simulator.device}."
        )

    if total_spectra <= 0:
        raise ValueError(
            "total_spectra must be > 0."
        )

    if batch_size <= 0:
        raise ValueError(
            "batch_size must be > 0."
        )

    generator = torch.Generator(
        device=simulator.device
    )

    generator.manual_seed(
        seed
    )

    # Warm-up:
    # initializes CUDA kernels, FFT plans, and allocator caches.
    for _ in range(
        warmup_batches
    ):
        warmup_batch = simulator.simulate(
            batch_size=batch_size,
            generator=generator,
        )

        del warmup_batch

    torch.cuda.synchronize(
        simulator.device
    )

    torch.cuda.reset_peak_memory_stats(
        simulator.device
    )

    generated = 0

    start = time.perf_counter()

    while generated < total_spectra:
        current_batch_size = min(
            batch_size,
            total_spectra - generated,
        )

        simulated_batch = simulator.simulate(
            batch_size=current_batch_size,
            generator=generator,
        )

        generated += current_batch_size

        # Do not retain previously simulated batches.
        del simulated_batch

    # GPU operations are asynchronous, so synchronization is
    # essential before stopping the timer.
    torch.cuda.synchronize(
        simulator.device
    )

    elapsed_seconds = (
        time.perf_counter()
        - start
    )

    spectra_per_second = (
        generated
        / elapsed_seconds
    )

    milliseconds_per_spectrum = (
        1000.0
        * elapsed_seconds
        / generated
    )

    peak_memory_gb = (
        torch.cuda.max_memory_allocated(
            simulator.device
        )
        / 1024**3
    )

    return {
        "generated": float(
            generated
        ),
        "batch_size": float(
            batch_size
        ),
        "elapsed_seconds": (
            elapsed_seconds
        ),
        "spectra_per_second": (
            spectra_per_second
        ),
        "milliseconds_per_spectrum": (
            milliseconds_per_spectrum
        ),
        "peak_memory_gb": (
            peak_memory_gb
        ),
    }


result = benchmark_spectrum_simulator(
    simulator=gpu_spectrum_simulator,
    total_spectra=1500_000,
    batch_size=3500,
)


print(
    f"Generated: "
    f"{int(result['generated']):,}"
)

print(
    f"Batch size: "
    f"{int(result['batch_size']):,}"
)

print(
    f"Elapsed: "
    f"{result['elapsed_seconds']:.3f} s"
)

print(
    f"Throughput: "
    f"{result['spectra_per_second']:,.0f} spectra/s"
)

print(
    f"Time per spectrum: "
    f"{result['milliseconds_per_spectrum']:.6f} ms"
)

print(
    f"Peak allocated GPU memory: "
    f"{result['peak_memory_gb']:.2f} GB"
)

In [ ]:
import time
import torch


def run_finite_stress_test(
    *,
    simulator: SpectrumSimulator,
    total_spectra: int = 1_000_000,
    batch_size: int = 4096,
    seed: int = 123456,
) -> None:
    if simulator.device.type != "cuda":
        raise ValueError(
            "The stress test requires a CUDA simulator, "
            f"but found {simulator.device}."
        )

    if total_spectra <= 0:
        raise ValueError(
            "total_spectra must be > 0."
        )

    if batch_size <= 0:
        raise ValueError(
            "batch_size must be > 0."
        )

    generator = torch.Generator(
        device=simulator.device
    )

    generator.manual_seed(
        seed
    )

    generated = 0
    batch_index = 0
    total_retries = 0

    discarded_batches_before = (
        simulator.discarded_batches
    )

    discarded_spectra_before = (
        simulator.discarded_spectra
    )

    torch.cuda.synchronize(
        simulator.device
    )

    start = time.perf_counter()

    try:
        while generated < total_spectra:
            current_batch_size = min(
                batch_size,
                total_spectra - generated,
            )

            batch = simulator.simulate(
                batch_size=current_batch_size,
                generator=generator,
            )

            # Final check on exactly the tensors that will be
            # passed to the network.
            input_valid = torch.isfinite(
                batch.network_input
            ).all()

            target_valid = torch.isfinite(
                batch.network_target
            ).all()

            scale_valid = torch.isfinite(
                batch.normalization_scale
            ).all()

            if not bool(
                input_valid
                & target_valid
                & scale_valid
            ):
                raise RuntimeError(
                    "A non-finite value reached the final "
                    "network input, target, or normalization "
                    "scale."
                )

            total_retries += (
                batch.retries_used
            )

            generated += current_batch_size
            batch_index += 1

            del batch

    except RuntimeError as error:
        torch.cuda.synchronize(
            simulator.device
        )

        elapsed = (
            time.perf_counter()
            - start
        )

        print(
            "STRESS TEST FAILED"
        )

        print(
            f"Completed spectra before failure: "
            f"{generated:,}"
        )

        print(
            f"Failed batch index: "
            f"{batch_index}"
        )

        print(
            f"Failed batch size: "
            f"{current_batch_size:,}"
        )

        print(
            f"Elapsed before failure: "
            f"{elapsed:.3f} s"
        )

        print(
            f"Successful internal retries before failure: "
            f"{total_retries}"
        )

        print(
            "Original error:"
        )

        print(
            repr(error)
        )

        return

    torch.cuda.synchronize(
        simulator.device
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    discarded_batches_during_test = (
        simulator.discarded_batches
        - discarded_batches_before
    )

    discarded_spectra_during_test = (
        simulator.discarded_spectra
        - discarded_spectra_before
    )

    print(
        f"Generated: {generated:,}"
    )

    print(
        f"Elapsed: {elapsed:.3f} s"
    )

    print(
        f"Throughput: "
        f"{generated / elapsed:,.0f} spectra/s"
    )

    print(
        "Non-finite final outputs: 0"
    )

    print(
        f"Internal retries used: "
        f"{total_retries}"
    )

    print(
        f"Discarded batches: "
        f"{discarded_batches_during_test}"
    )

    print(
        f"Discarded spectra: "
        f"{discarded_spectra_during_test:,}"
    )

In [ ]:
run_finite_stress_test(
    simulator=gpu_spectrum_simulator,
    total_spectra=1_000_000,
    batch_size=3500,
    seed=123456,
)

In [ ]:
import inspect
import walinet.training_data.spectrum_simulator as spectrum_simulator_module


print(
    "Imported file:",
    spectrum_simulator_module.__file__,
)

print(
    "Constructor:",
    inspect.signature(
        spectrum_simulator_module
        .SpectrumSimulator
        .__init__
    ),
)

print(
    "PreparedSpectrumBatch exists:",
    hasattr(
        spectrum_simulator_module,
        "PreparedSpectrumBatch",
    ),
)

In [ ]:
from walinet.training_data.spectrum_simulator import (
    SpectrumSimulator,
)

gpu_spectrum_simulator = SpectrumSimulator(
    pool=gpu_resources.train,
    metabolite_simulator=gpu_metabolite_simulator,
    config=simulation_cfg,
    max_retries=3,
)

In [ ]:


generator = torch.Generator(
    device=gpu_spectrum_simulator.device
)

generator.manual_seed(
    123456
)


batch = gpu_spectrum_simulator.simulate(
    batch_size=4096,
    generator=generator,
)


inputs = batch.network_input
targets = batch.network_target

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch


# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------
n_spectra = 40
n_columns = 3

# Options:
#     "real"
#     "imag"
#     "magnitude"
display_mode = "real"

# None:
#     display complete spectral range
#
# Typical metabolite range:
#     (4.5, 0.2)
ppm_window = None

ppm_reference = 4.68


# ---------------------------------------------------------
# Simulate final trainer-ready batch
# ---------------------------------------------------------
plot_generator = torch.Generator(
    device=gpu_spectrum_simulator.device
)

plot_generator.manual_seed(
    20260715
)


plot_batch = gpu_spectrum_simulator.simulate(
    batch_size=n_spectra,
    generator=plot_generator,
)


# ---------------------------------------------------------
# Reconstruct complex spectra from the actual network tensors
#
# network_input:
#     (B, 2, T)
#
# channel 0:
#     real
#
# channel 1:
#     imaginary
# ---------------------------------------------------------
input_spectra = torch.complex(
    plot_batch.network_input[:, 0, :],
    plot_batch.network_input[:, 1, :],
)

target_spectra = torch.complex(
    plot_batch.network_target[:, 0, :],
    plot_batch.network_target[:, 1, :],
)


# Verify that the network representation is exactly equivalent
# to the normalized complex representation.
torch.testing.assert_close(
    input_spectra,
    plot_batch.normalized_input_spectra,
)

torch.testing.assert_close(
    target_spectra,
    plot_batch.normalized_target_spectra,
)


# Every final input spectrum must have max(abs(input)) == 1.
input_maxima = torch.amax(
    torch.abs(
        input_spectra
    ),
    dim=-1,
)

torch.testing.assert_close(
    input_maxima,
    torch.ones_like(
        input_maxima
    ),
    atol=1e-5,
    rtol=1e-5,
)


# ---------------------------------------------------------
# Frequency and ppm axes
# ---------------------------------------------------------
n_timepoints = (
    simulation_cfg
    .acquisition
    .n_timepoints
)

bandwidth_hz = (
    simulation_cfg
    .acquisition
    .bandwidth_hz
)

nmr_frequency_hz = (
    simulation_cfg
    .acquisition
    .nmr_frequency_hz
)

hz_per_ppm = (
    nmr_frequency_hz
    / 1e6
)

dwell_time_seconds = (
    1.0
    / bandwidth_hz
)

frequency_axis_hz = np.fft.fftshift(
    np.fft.fftfreq(
        n_timepoints,
        d=dwell_time_seconds,
    )
)

ppm_axis = (
    ppm_reference
    - frequency_axis_hz
    / hz_per_ppm
)


# ---------------------------------------------------------
# Move only the small plotting batch to CPU
# ---------------------------------------------------------
input_np = (
    input_spectra
    .detach()
    .cpu()
    .numpy()
)

target_np = (
    target_spectra
    .detach()
    .cpu()
    .numpy()
)

acquired_lengths = (
    plot_batch
    .acquired_n_timepoints
    .detach()
    .cpu()
    .numpy()
)

snr_values = (
    plot_batch
    .snr
    .detach()
    .cpu()
    .numpy()
)

water_scaling = (
    plot_batch
    .water_scaling
    .detach()
    .cpu()
    .numpy()
)

lipid_scaling = (
    plot_batch
    .lipid_scaling
    .detach()
    .cpu()
    .numpy()
)


# ---------------------------------------------------------
# Select displayed signal component
# ---------------------------------------------------------
def select_display_component(
    spectra: np.ndarray,
    mode: str,
) -> np.ndarray:
    if mode == "real":
        return spectra.real

    if mode == "imag":
        return spectra.imag

    if mode == "magnitude":
        return np.abs(
            spectra
        )

    raise ValueError(
        "display_mode must be one of "
        "{'real', 'imag', 'magnitude'}."
    )


input_display = select_display_component(
    input_np,
    display_mode,
)

target_display = select_display_component(
    target_np,
    display_mode,
)


# ---------------------------------------------------------
# Raster plot
# ---------------------------------------------------------
n_rows = math.ceil(
    n_spectra
    / n_columns
)

figure, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(
        5.0 * n_columns,
        3.1 * n_rows,
    ),
    squeeze=False,
    sharex=True,
)


for spectrum_index, axis in enumerate(
    axes.flat
):
    if spectrum_index >= n_spectra:
        axis.axis(
            "off"
        )
        continue

    axis.plot(
        ppm_axis,
        input_display[
            spectrum_index
        ],
        linewidth=0.9,
        label="Input",
    )

    axis.plot(
        ppm_axis,
        target_display[
            spectrum_index
        ],
        linewidth=0.9,
        alpha=0.8,
        label="Target",
    )

    axis.axhline(
        0.0,
        linewidth=0.5,
        alpha=0.5,
    )

    if ppm_window is None:
        axis.set_xlim(
            float(
                ppm_axis.max()
            ),
            float(
                ppm_axis.min()
            ),
        )

    else:
        axis.set_xlim(
            ppm_window
        )

    axis.set_title(
        (
            f"Sample {spectrum_index} | "
            f"FID={acquired_lengths[spectrum_index]} | "
            f"SNR={snr_values[spectrum_index]:.2f}\n"
            f"Water={water_scaling[spectrum_index]:.2f} | "
            f"Lipid={lipid_scaling[spectrum_index]:.2f}"
        ),
        fontsize=9,
    )

    axis.grid(
        alpha=0.2
    )


handles, labels = (
    axes[0, 0]
    .get_legend_handles_labels()
)

figure.legend(
    handles,
    labels,
    loc="upper right",
)

figure.supxlabel(
    "Chemical shift [ppm]"
)

figure.supylabel(
    f"Normalized {display_mode}"
)

figure.suptitle(
    (
        "Final normalized simulator output\n"
        "Input: metabolites + water + lipids + noise | "
        "Target: metabolites + noise"
    )
)

figure.tight_layout(
    rect=(
        0.02,
        0.02,
        0.98,
        0.95,
    )
)

plt.show()